# 02 - Mask classification on LAS tiles (Site 11)

Sequential processing to classify LAS points using raster masks.

- Crops to raster bounds
- Applies mask-driven classification (values 64-68)
- Preserves original scale/offset and metadata

In [33]:
from pathlib import Path

# --- CONFIGURATION: Site 11 (DD0011) ---
SITE_NAME = 'site2_1_DD0011'

# Input directories: Imagery
LAS_DIR = Path('/data/calperumResearch/site2_1_DD0011/lidar/clipped/raster/coreg/icp')
AOI_RASTER_DIR = Path('/data/calperumResearch/site2_1_DD0011/imagery/inputs/predictors/tiles_3072/stacked')
MASK_DIR = Path('/data/calperumResearch/site2_1_DD0011/imagery/inputs/masks/tiles_3072/structure_class/annotation_raster/downsample_5cm_nearest')


# If LAS_DIR was set to a specific LAS file path, normalize to its parent directory
if LAS_DIR.is_file():
    print(f"Note: LAS_DIR points to a file. Using parent folder: {LAS_DIR.parent}")
    LAS_DIR = LAS_DIR.parent

# Pattern for LAS tiles produced by fvclip
RASTER_PATTERN = 'tiles_multispectral*.las'

# Output directory for mask classification
OUT_DIR = Path('/data/calperumResearch/site2_1_DD0011/imagery/clipped/mask')
# OUT_DIR.mkdir(parents=True, exist_ok=True)

print('--- Configuration ---')
print(f'Raster dir  : {AOI_RASTER_DIR}')
print(f'Mask dir    : {MASK_DIR}')
print(f'LAS dir     : {LAS_DIR}')
print(f'Pattern     : {RASTER_PATTERN}')
print(f'Output dir  : {OUT_DIR}')
print('----------------------')

--- Configuration ---
Raster dir  : /data/calperumResearch/site2_1_DD0011/imagery/inputs/predictors/tiles_3072/stacked
Mask dir    : /data/calperumResearch/site2_1_DD0011/imagery/inputs/masks/tiles_3072/structure_class/annotation_raster/downsample_5cm_nearest
LAS dir     : /data/calperumResearch/site2_1_DD0011/lidar/clipped/raster/coreg/icp
Pattern     : tiles_multispectral*.las
Output dir  : /data/calperumResearch/site2_1_DD0011/imagery/clipped/mask
----------------------


In [31]:
# --- Sequential mask classification (matching coreg pattern) ---
import json
import pdal
import rasterio
import laspy
from tqdm import tqdm

# Find all LAS files
las_files = sorted(LAS_DIR.glob(RASTER_PATTERN))

if not las_files:
    print(f"❌ No LAS files found in {LAS_DIR}")
else:
    print(f"Found {len(las_files)} LAS files to process")
    
    # Process each file
    for input_las in tqdm(las_files, desc="Processing tiles"):
        # Extract tile ID (e.g., "tiles_multispectral.33.las" -> "33")
        tile_id = input_las.stem.split(".")[-1]
        
        # Construct corresponding raster and mask paths
        aoi_raster_5cm = AOI_RASTER_DIR / f"tiles_multispectral.{tile_id}.tif"
        aoi_mask_tif_5cm = MASK_DIR / f"mask_structure_3072.{tile_id}.tif"
        
        # Validate files exist
        if not aoi_raster_5cm.exists():
            print(f"⚠️  Skipping tile {tile_id}: raster not found")
            continue
        if not aoi_mask_tif_5cm.exists():
            print(f"⚠️  Skipping tile {tile_id}: mask not found")
            continue
        
        # Output file
        raster_name = aoi_raster_5cm.stem
        output_classified_las = OUT_DIR / f"mask_{raster_name}.las"
        
        # Skip if already processed
        if output_classified_las.exists():
            continue
        
        # --- Preserve original scale/offset ---
        in_las = laspy.read(input_las)
        sx, sy, sz = in_las.header.scales
        ox, oy, oz = in_las.header.offsets
        minor_ver = int(in_las.header.version.minor)
        
        # === Read raster bounds ===
        with rasterio.open(aoi_raster_5cm) as src:
            target_epsg = src.crs.to_epsg()
            minx, miny, maxx, maxy = src.bounds
        
        bounds_str = f"([{minx},{maxx}],[{miny},{maxy}])"
        
        # === Build the PDAL pipeline ===
        pipeline = {
            "pipeline": [
                {"type": "readers.las", "filename": str(input_las)},
                {"type": "filters.crop", "bounds": bounds_str},
                {"type": "filters.colorization", "raster": str(aoi_mask_tif_5cm), "dimensions": "Classification"},
                {"type": "filters.range", "limits": "Classification[64:68]"},
                {
                    "type": "writers.las",
                    "filename": str(output_classified_las),
                    "minor_version": minor_ver,
                    "extra_dims": "all",
                    "scale_x": sx, "scale_y": sy, "scale_z": sz,
                    "offset_x": ox, "offset_y": oy, "offset_z": oz,
                    "forward": "all"
                }
            ]
        }
        
        # Execute pipeline
        try:
            pipe = pdal.Pipeline(json.dumps(pipeline))
            n = pipe.execute()
            print(f"✅ Tile {tile_id}: {n:,} points → {output_classified_las.name}")
        except Exception as e:
            print(f"❌ Error processing tile {tile_id}: {e}")
    
    print(f"\n🎉 All processing complete.")

❌ No LAS files found in /data/calperumResearch/site2_1_DD0011/lidar/clipped/raster/coreg


In [35]:
# --- Inspect one output LAS ---
import laspy, numpy as np

# pick one output (first matching pattern). Support both naming styles.
candidates = sorted(OUT_DIR.glob(RASTER_PATTERN))
mask_named = sorted(OUT_DIR.glob('mask_tiles_multispectral*.las'))
candidates = mask_named or candidates
sample_out = candidates[0] if candidates else None
print(f'Inspecting: {sample_out}')
if sample_out and sample_out.exists():
    las = laspy.read(sample_out)
    print(f'Total points: {len(las.points):,}')
    print('Bounding box:')
    print(f'  X: {las.header.mins[0]:.2f} → {las.header.maxs[0]:.2f}')
    print(f'  Y: {las.header.mins[1]:.2f} → {las.header.maxs[1]:.2f}')
    print(f'  Z: {las.header.mins[2]:.2f} → {las.header.maxs[2]:.2f}')
    print(f'Point format: {las.header.point_format}')
    print(f'Extra dims : {[dim.name for dim in las.point_format.extra_dimensions]}')
    if hasattr(las, 'classification'):
        cls = np.asarray(las.classification, dtype=np.uint8)
        unique, counts = np.unique(cls, return_counts=True)
        print('Classification counts:')
        for u, c in zip(unique, counts):
            print(f'  class {int(u)}: {int(c):,}')
else:
    print('No outputs found yet. Ensure fvclip produced LAS tiles and paths match.')

Inspecting: None
No outputs found yet. Ensure fvclip produced LAS tiles and paths match.


In [36]:
# --- View original raster (bands 5/3/1) ---
import rasterio, numpy as np, matplotlib.pyplot as plt

# choose a raster tile matching sample_out if available
if sample_out:
    tile_id = sample_out.stem.split('.')[-1]
    tile_path = AOI_RASTER_DIR / f'tiles_multispectral.{tile_id}.tif'
else:
    # fallback: first raster
    rlist = sorted(AOI_RASTER_DIR.glob('tiles_multispectral.*.tif'))
    tile_path = rlist[0] if rlist else None

if tile_path is None or not tile_path.exists():
    raise RuntimeError('Could not locate a matching raster tile for visualization.')

with rasterio.open(tile_path) as src:
    if src.count < 5:
        raise RuntimeError(f"Raster has {src.count} bands; need >= 5 for 5/3/1 RGB.")
    b5 = src.read(5).astype(np.float32)
    b3 = src.read(3).astype(np.float32)
    b1 = src.read(1).astype(np.float32)
    def norm_band(b):
        lo, hi = np.percentile(b[np.isfinite(b)], [2, 98])
        b = np.clip((b - lo) / (hi - lo + 1e-6), 0, 1)
        b[~np.isfinite(b)] = 0
        return b
    rgb = np.stack([norm_band(b5), norm_band(b3), norm_band(b1)], axis=-1)
    extent = (src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top)

plt.figure(figsize=(8, 8))
plt.imshow(rgb, extent=extent, origin='upper')
plt.title('Multispectral chunk raster (bands: NIR-842 nm, Red-668 nm, Blue-475 nm)')
plt.xlabel('X'); plt.ylabel('Y')
plt.tight_layout(); plt.show()

RuntimeError: Could not locate a matching raster tile for visualization.